In [1]:
!pip install arxiv


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
##For Extracting wikipedia data
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)
wiki.name


'wikipedia'

In [3]:
#For webpage

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings                        # ✅ updated import
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ✅ fixed typo

# 1. Load webpage
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

# 2. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

# 3. Embed and store in FAISS
db = FAISS.from_documents(
    documents,
    OllamaEmbeddings(model="nomic-embed-text")  
)

# 4. Retriever
retriever = db.as_retriever()
retriever


USER_AGENT environment variable not set, consider setting it to identify your requests.


VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000280A75678C0>, search_kwargs={})

In [5]:
#creating a tool for retriever

from langchain_core.tools.retriever import create_retriever_tool  
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",                                        
    "Search for information about LangSmith. For any question about LangSmith, you must use this tool!"
)
retriever_tool.name


'langsmith_search'

In [23]:
#for arxiv ->free, open-access research paper repository
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200, arxiv_search_sleep_time=5   # ← add 5 second delay between requests
)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [7]:
tools=[wiki,arxiv,retriever_tool]

In [11]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2")

#for agent in openai we require first prompt,llm ,tools and then we gonana create an agent and then agent executer
from langchain.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)

#agent executer
frim langchain.agents import AgentExecuter
agent_executer=AgentExecuter(agent=agent,tools=tools,verbose=True)


In [18]:
#Agents
from langgraph.prebuilt import create_react_agent
agent_executor = create_react_agent(llm, tools)


C:\Users\Piyush Agarwal\AppData\Local\Temp\ipykernel_17112\4272426177.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


In [ ]:
agent_executor.invoke({"messages": [("user", "Tell me about machine learning")]})
#invoke in output tells it goes to wikipedia to process

{'messages': [HumanMessage(content='Tell me about machine learning', additional_kwargs={}, response_metadata={}, id='188ab0ca-1dca-4b03-81a2-17fdb589e6c7'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-05-15T06:14:09.5610794Z', 'done': True, 'done_reason': 'stop', 'total_duration': 894192200, 'load_duration': 87661800, 'prompt_eval_count': 334, 'prompt_eval_duration': 439884700, 'eval_count': 18, 'eval_duration': 347738200, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019e2a45-5a18-7c20-aeab-45b3d950400e-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'machine learning'}, 'id': '962b0447-0f01-4d84-a8e6-5e23afca71fe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 334, 'output_tokens': 18, 'total_tokens': 352}),
  ToolMessage(content='Page: Machine learning\nSummary: Machine learning (ML) is a field of study in artificial intelligence concerned with

In [20]:
response = agent_executor.invoke({"messages": [("user", "Tell me about langsmith")]})
print(response["messages"][-1].content)

Langsmith is a tool that helps you build, debug, and deploy AI agents and Large Language Model (LLM) applications. It provides a platform-agnostic approach, allowing users to integrate it with various frameworks and providers.

With Langsmith, you can:

1. **Trace requests**: Gain visibility into every step your application takes to debug faster and improve reliability.
2. **Evaluate outputs**: Measure and track quality over time to ensure your AI applications are consistent and trustworthy.
3. **Prompt engineering**: Optimize prompts for better model performance.
4. **Agent deployment**: Deploy your agents as Agent Servers, ready to scale in production.

Langsmith also provides features such as:

1. **Workspace setup**: Create a workspace to manage your projects and users.
2. **User access control**: Manage user permissions and access to your workspaces.
3. **Billing and usage**: Track usage and billing for your Langsmith account.
4. **Audit logs**: Monitor audit logs for your organiz

In [ ]:
agent_executor.invoke({"messages": [("user", "whats the paper 1605.08386 about?")]})
#invoke in optput tells it goes to arxiv to process

{'messages': [HumanMessage(content='whats the paper 1605.08386 about?', additional_kwargs={}, response_metadata={}, id='bb689dbb-45f7-4deb-8815-f5a9525c97dd'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-05-15T06:19:21.2012378Z', 'done': True, 'done_reason': 'stop', 'total_duration': 757279200, 'load_duration': 108389300, 'prompt_eval_count': 341, 'prompt_eval_duration': 217332600, 'eval_count': 21, 'eval_duration': 409494200, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019e2a4a-1bf9-7fe3-96a2-d7a0fc45da4a-0', tool_calls=[{'name': 'arxiv', 'args': {'query': '1605.08386'}, 'id': '68414293-59d1-48cd-bcae-cfd509d3dae6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 341, 'output_tokens': 21, 'total_tokens': 362}),
  ToolMessage(content='Published: 2016-05-26\nTitle: Heat-bath random walks with Markov bases\nAuthors: Caprice Stanley, Tobias Windisch\nSummary

In [25]:
response = agent_executor.invoke({"messages": [("user", "whats the paper 1605.08386 about?")]})
print(response["messages"][-1].content)

The paper "1605.08386" is titled "Heat-bath random walks with Markov bases". The authors, Caprice Stanley and Tobias Windisch, study graphs on lattice points with edges coming from a finite set of allowed moves.

In simpler terms, the researchers explored how random walks (a series of steps in a specific order) move on a grid-like structure using a heat-bath process. They also introduced the concept of Markov bases to define these graph structures.

Would you like me to elaborate further or provide more context?
